In [1]:
import os
from pathlib import Path

import librosa
import torch.cuda
from IPython.display import Audio
from dotenv import load_dotenv
from faster_whisper import WhisperModel

from src.metrics import calculate_csr_metrics

load_dotenv()

/home/dom/GitRepos/multilingual-transcription/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
whisper = WhisperModel("small", device=device, compute_type="auto", use_auth_token=os.environ["HUGGINGFACE_TOKEN"])
whisper.model.device, whisper.model.compute_type

('cuda', 'int8_float16')

In [4]:
audio_path = Path("../data/example_2/audio.mp3")
waveform, sample_rate = librosa.load(audio_path, sr=16000)
Audio(waveform, rate=sample_rate)

In [5]:
segments, info = whisper.transcribe("../data/example_2/audio.mp3", beam_size=5)
segments = list(segments)

transcription = ""

for segment in segments:
    text = segment.text.strip()
    transcription += text + "\n"

print(transcription)

with open("../data/example_3/transcription.txt", "r") as f:
    reference_transcription = f.read()

crs_metrics = calculate_csr_metrics(transcription, reference_transcription)
print(crs_metrics)

Hello, I'm Jane. Is this seat free?
Cześć Jane, nam na imię Michał. I jasne, możesz usiąść.
Thanks. It's my first time in Kraków.
Naprawdę? Jak ci się podoba miasto?
I love it so far. The old town is beautiful.
Stare miasto jest bardzo popularne wśród turystów. Byłaś już na rynku?
Yes, I was there this morning. I saw St. Mary's Basilica.
O, Kościół Mariacki. Codziennie grają tam hajnał zwierzę.
Great. And can you recommend some Polish food?
Oczywiście. Powinnaś spróbować pierogów i żurku.
I know pierogi. They're delicious.
A próbowałaś już pączków?
Not yet. What are they?
To polskie donac. Bardzo słodkie, ale świetna.
Sounds dangerous for my diet.
Polska kuchnia często jest niebezpieczna dla diety.


         CSR results
--------------------------------
 WER (Word Error Rate):    0.11
 MER (Match Error Rate):   0.11
 WDER (Diarization Error): -1.00



In [6]:
segments, info = whisper.transcribe("../data/example_3/audio.mp3", beam_size=5)
segments = list(segments)

transcription = ""

for segment in segments:
    text = segment.text.strip()
    transcription += text + "\n"

print(transcription)

with open("../data/example_3/transcription.txt", "r") as f:
    reference_transcription = f.read()

crs_metrics = calculate_csr_metrics(transcription, reference_transcription)
print(crs_metrics)

Hello, I'm Josh. Is this seat free?
Cześć, Josh. Mam na imię Maya. I jasne, możesz usiąść.
Thanks. It's my first time in Klaku.
Naprawdę? Jak ci się podoba miasto?
I love it so far. The old town is beautiful.
Stare miasto jest bardzo popularne wśród turystów. Byłaś już na rynku?
Yes, I was there this morning. I saw St. Mary's Basilica.
O, kościół Moriacki. Codziennie grają tam hejnał zwierzy.
Great. And can you recommend some Polish food?
Oczywiście. Powinnaś spróbować pierogów i żurku.
I know pierogi. They're delicious.
A, próbowałaś już pątków?
Not yet. What are they?
To polskie doughnuts. Bardzo słodkie, ale świetne.
Sounds dangerous for my diet.
Polska kuchnia często jest niebezpieczna dla diety.
Haha, I can believe that.


         CSR results
--------------------------------
 WER (Word Error Rate):    0.05
 MER (Match Error Rate):   0.05
 WDER (Diarization Error): -1.00

